# Sanitised Data Pipeline

This notebook preserves `data/Survey_Results_UC.csv` unchanged and writes cleaned/imputed numeric copies plus network, validation, and report outputs under `outputs/sanitised_data/`.

Important statistical choices:

- `No Comments` is missing (`NC`), not Neutral.
- Structural blank cells are missing (`STRUCTURAL_BLANK`).
- Spearman rank correlation is the primary ordinal/rank-based network association.
- Pearson correlation is not used as the primary network edge.
- The five entirely empty submissions are excluded before analysis/export.
- Main final datasets use numeric response values `-2, -1, 0, 1, 2`; a separate label copy is kept for readability.
- Missing responses are imputed with question-wise regularized proportional-odds ordinal logistic regression.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.ordinal_pipeline import PipelineConfig, run_pipeline

## Configurable Parameters

In [2]:
config = PipelineConfig(
    raw_csv=PROJECT_ROOT / "data" / "Survey_Results_UC.csv",
    output_dir=PROJECT_ROOT / "outputs" / "sanitised_data",
    min_pairwise_n=50,
    edge_threshold=0.30,
    test_size=0.20,
    random_state=42,
    validation_repeats=5,
    validation_mask_rate=0.12,
    structural_validation_rows=40,
    ordinal_l2=1.0,
    max_iter=500,
)

## Run Complete Workflow

Workflow:

`Raw questionnaire -> Missingness identification -> Ordinal encoding -> Pairwise Spearman correlation -> 60-node questionnaire network -> Network visualization/statistics -> Question-wise ordinal regression -> Artificial-masking validation -> Impute genuinely missing responses`

In [3]:
results = run_pipeline(config)

Sanitised data pipeline complete
Raw respondents: 96
Excluded entirely empty respondents: 5
Respondents analysed/exported: 91
Questions: 60
Structural blanks: 205
NC cells: 37
Usable respondents: 91
Network nodes: 60
Network edges: 1770
Chosen edge threshold: 0.3
Mean |correlation|: 0.183
Median |correlation|: 0.164
Ordinal validation accuracy: 0.476
Ordinal validation MAE: 0.660
Cells imputed: 242
Outputs written to: /Users/arijeetpaul/Documents/dpcn_a1_opinion_network/outputs/sanitised_data


## Summary Object

In [4]:
results["summary"]

{'respondents': 91,
 'questions': 60,
 'no_structural_blank_rows': 85,
 'fully_observed_likert_rows': 68,
 'stopped_after_technology_rows': 4,
 'stopped_partway_environment_rows': 2,
 'entirely_empty_rows': 0,
 'structural_blank_cells': 205,
 'nc_cells': 37,
 'nc_users': 17,
 'usable_respondents': 91,
 'raw_respondents': 96,
 'excluded_entirely_empty_rows': 5,
 'raw_structural_blank_cells': 505,
 'raw_nc_cells': 37,
 'raw_nc_users': 17,
 'node_count': 60,
 'edge_count': 1770,
 'density': 1.0,
 'mean_degree': 59.0,
 'weighted_average_correlation': 0.15263238206353943,
 'mean_abs_correlation': 0.1828779688049621,
 'median_abs_correlation': 0.16401342750003525,
 'positive_edge_count': 1475,
 'negative_edge_count': 295,
 'edge_threshold': 0.3,
 'thresholded_edge_count': 328,
 'cells_imputed': 242,
 'validation_accuracy': 0.47600730090218824,
 'validation_mae': 0.6596181059692852}